In [1]:
import sqlite3

In [16]:
class Catalog:
    def __init__(self, dbfile):
        print("Creating Catalog")
        self.conn = sqlite3.connect(dbfile)
        self.cursor = self.conn.cursor()
        self.setup()

        
    def setup(self):

        self.cursor.execute("""
        CREATE TABLE IF NOT EXISTS products (
            id INTEGER PRIMARY KEY,
            brand TEXT NOT NULL,
            name TEXT NOT NULL,
            variant TEXT NOT NULL,
            UNIQUE(brand, name, variant)
        );
        """)

        self.cursor.execute("""
        CREATE TABLE IF NOT EXISTS prices (
        id INTEGER PRIMARY KEY,
        product_id INTEGER NOT NULL,
        price REAL NOT NULL,
        timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,

        FOREIGN KEY(product_id) REFERENCES products(id) ON DELETE CASCADE
        );
        """)
        
        self.cursor.execute("""
        CREATE TABLE IF NOT EXISTS product_urls (
        id INTEGER PRIMARY KEY,
        product_id INTEGER NOT NULL,
        url TEXT NOT NULL,
        
        FOREIGN KEY(product_id) REFERENCES products(id) ON DELETE CASCADE
        );
        """)


        self.conn.commit()

        
    def addProduct(self, brand:str, name: str, variant:str, urls:list):
        try:
            self.cursor.execute(
                """
                INSERT INTO products (brand, name, variant)
                VALUES (?, ?, ?)
                """,
                (brand, name, variant),
            )

            product_id = self.cursor.lastrowid

            for url in urls:
                self.cursor.execute(
                    """
                    INSERT INTO product_urls (product_id, url)
                    VALUES (?, ?)
                    """,
                    (product_id, url),
                )

            self.conn.commit()

        except Exception:
            self.conn.rollback()
            raise

In [17]:
cat = Catalog('catalog.db')

Creating Catalog


In [18]:
cat.addProduct("Lelo", "Liv 3", "Deep Rose", ["https://www.lelo.com/liv-3", 
                                              "https://www.amazon.com/LELO-App-Controlled-Vibrator-Waterproof-Vibrators/dp/B0D9YS4497"])

In [19]:
PriceClassMap = {"lelo.com": "EmphasizedPrice"}

In [18]:
# from playwright.sync_api import sync_playwright
from playwright.async_api import async_playwright

async def get_price(url):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        await page.goto(url)

        
        cls = '[class*="EmphasizedPrice"]'
        await page.wait_for_selector(cls)
        
        elements = await page.locator(cls).all()
        
        for el in elements:
            text = await el.text_content()
            print(text)
        await browser.close()

await get_price("https://www.lelo.com/gigi-3")

USD 116.07
